In [32]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
# 把 deeplearning 目录加入模块搜索路径
d2l_dir = Path(r"C:\Users\abc\Documents\GitHub\vscodepractice\deeplearning")
sys.path.insert(0, str(d2l_dir))
import math
import numpy as np
import torch
from torch import nn
import d2l_local as d2l

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
def corr2d(X, K):  #@save
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y


In [34]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

In [35]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

kernel_size = (2, 2)
conv2d = Conv2D(kernel_size)
print(conv2d.weight)
print(conv2d.bias)

Parameter containing:
tensor([[0.6917, 0.0879],
        [0.0156, 0.9830]], requires_grad=True)
Parameter containing:
tensor([0.], requires_grad=True)


In [36]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [37]:
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)
print(Y)

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])


In [38]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = nn.Conv2d(1,1, kernel_size=(1, 2), bias=False)

# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # 学习率

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # 迭代卷积核
    with torch.no_grad():
        conv2d.weight -= lr * conv2d.weight.grad
    #conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 11.175
epoch 4, loss 3.357
epoch 6, loss 1.170
epoch 8, loss 0.445
epoch 10, loss 0.176


In [39]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 1.0312, -0.9454]])

In [40]:
print(torch.__version__)

X_test = torch.tensor([
    [0., 1., 2.],
    [3., 4., 5.],
    [6., 7., 8.]
])

my_conv = Conv2D((2, 2))

Y_test = my_conv(X_test)
loss = Y_test.sum()

loss.backward()

print("输出：")
print(Y_test)

print("卷积核梯度：")
print(my_conv.weight.grad)

print("偏置梯度：")
print(my_conv.bias.grad)

2.10.0+cu128
输出：
tensor([[ 3.5580,  5.3538],
        [ 8.9454, 10.7412]], grad_fn=<AddBackward0>)
卷积核梯度：
tensor([[ 8., 12.],
        [20., 24.]])
偏置梯度：
tensor([4.])
